Tile each city bounding box into regions of interest (ROIs) of about 1 km x 1 km and write the ROI grid (All_Cities_ROI_Grid1.csv).

In [16]:
import pandas as pd
import numpy as np

In [40]:
# Define paths
excel_file_path = '<ROI_GRID_DIR>/cities_list_bbox copy.xlsx'
output_all_cities_path = '<ROI_GRID_DIR>/All_Cities_ROI_Grid.csv'
output_roi_count_path = '<ROI_GRID_DIR>/Cities_ROI_Count.csv'


In [42]:
bbox_data = pd.read_excel(excel_file_path)

In [44]:
bbox_data

In [46]:
All_ROI_df = pd.DataFrame(columns=['City', 'Country', 'Max_Latitude', 'Min_Longitude', 'Min_Latitude', 'Max_Longitude'])
ROI_count_df = pd.DataFrame(columns=['City', 'Country', 'ROI_Count'])

In [48]:
def latLonToMeters(lat1, lon1, lat2, lon2):
    R = 6371000  # Radius of the Earth in meters
    lat1, lon1, lat2, lon2 = np.radians([lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distance = R * c
    return distance

In [50]:
# Loop through all cities in the bbox_data
for city_idx, row in bbox_data.iterrows():
    city_name = row['City']
    country_name = row['Country']

    minLat = float(row['Min_Latitude'])
    maxLat = float(row['Max_Latitude'])
    minLon = float(row['Min_Longitude'])
    maxLon = float(row['Max_Longitude'])

    dyLat = latLonToMeters(minLat, minLon, minLat + 0.01, minLon)
    latStep = 1000 / dyLat * 0.01  # Scale step size for 1000 meters in latitude

    dxLon = latLonToMeters(minLat, minLon, minLat, minLon + 0.01)
    lonStep = 1000 / dxLon * 0.01  # Scale step size for 1000 meters in longitude

    numRows = int(np.ceil((maxLat - minLat) / latStep))
    numCols = int(np.ceil((maxLon - minLon) / lonStep))
    ROI_count = 0  # Initialize ROI count for the current city

    tempROI_list = []

    for i in range(numRows):
        for j in range(numCols):
            roi_minLat = minLat + i * latStep
            roi_maxLat = roi_minLat + latStep
            roi_minLon = minLon + j * lonStep
            roi_maxLon = roi_minLon + lonStep

            tempROI_list.append([city_name, country_name, roi_maxLat, roi_minLon, roi_minLat, roi_maxLon])
            ROI_count += 1

    All_ROI_df = pd.concat([All_ROI_df, pd.DataFrame(tempROI_list, columns=['City', 'Country', 'Max_Latitude', 'Min_Longitude', 'Min_Latitude', 'Max_Longitude'])], ignore_index=True)

    # Use pd.concat instead of append
    new_row = pd.DataFrame([{'City': city_name, 'Country': country_name, 'ROI_Count': ROI_count}])
    ROI_count_df = pd.concat([ROI_count_df, new_row], ignore_index=True)


In [54]:
All_ROI_df.to_csv('All_Cities_ROI_Grid.csv', index=False)
ROI_count_df.to_csv('Cities_ROI_Count.csv', index=False)


In [56]:
All_ROI_df
